In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, f1_score
from imblearn.pipeline import Pipeline as ImbPipeline  # Use imblearn's Pipeline
import matplotlib.pyplot as plt
import warnings
from alive_progress import alive_bar
from contextlib import contextmanager
from joblib.parallel import BatchCompletionCallBack
import threading

# Import oversampling techniques from imbalanced-learn
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Load the dataset
# Replace the file path with your actual path
data = pd.read_excel("class123_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Define the feature indexes to be used as predictors
# Note: Pandas uses 0-based indexing
feature_indexes = [138, 244, 137, 224, 118, 103, 230, 42, 78, 204, 13, 33, 86, 123, 24, 3, 5, 21, 183, 126, 119, 65, 201, 30, 253, 122, 88, 49, 210, 80, 214, 217, 52, 206, 173, 90, 44, 59, 71, 229, 25, 157, 68, 247, 227, 182, 28]

# Select the specified features using .iloc
X_selected = X.iloc[:, feature_indexes]

# Create a pipeline with the sampler and classifier
pipeline = ImbPipeline([
    ('sampler', RandomOverSampler()),  # Placeholder, will be set by GridSearchCV
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define the extended parameter grid as a list of dictionaries
param_grid = [
    {
        # Include options with and without a sampler
        'sampler': [SMOTE(), ADASYN(), RandomOverSampler()],
        # Only include sampling_strategy if sampler is not None
        'sampler__sampling_strategy': ['auto', 0.5, 0.75, 1.0],

        # Classifier hyperparameters
        'classifier__n_estimators': [150, 200, 250, 300],  # Added 300 for more options
        'classifier__criterion': ['gini', 'entropy'],  # Added 'entropy' criterion for diversity
        'classifier__max_depth': [None] + list(range(5, 30, 10)),  # Depth extended to 45
        'classifier__min_samples_split': [2, 3, 5, 7],  # Added 2 as the default minimum split
        'classifier__min_samples_leaf': [1, 2, 3],  # Added 1 as the default minimum leaf size
        'classifier__max_features': ['log2', 'sqrt'],  # Added 'sqrt' for variety in feature selection
        'classifier__bootstrap': [False, True],  # Included True to test with bootstrapping
        'classifier__class_weight': [None, 'balanced']  # Added class weighting options
    },
    {
        'sampler': [None],

        # Classifier hyperparameters
        'classifier__n_estimators': [150, 200, 250, 300],  # Added 300 for more options
        'classifier__criterion': ['gini', 'entropy'],  # Added 'entropy' criterion for diversity
        'classifier__max_depth': [None] + list(range(5, 30, 10)),  # Depth extended to 45
        'classifier__min_samples_split': [2, 3, 5, 7],  # Added 2 as the default minimum split
        'classifier__min_samples_leaf': [1, 2, 3],  # Added 1 as the default minimum leaf size
        'classifier__max_features': ['log2', 'sqrt'],  # Added 'sqrt' for variety in feature selection
        'classifier__bootstrap': [False, True],  # Included True to test with bootstrapping
        'classifier__class_weight': [None, 'balanced']  # Added class weighting options
    }
]

# Define the scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'f1': 'f1'
}

# Define the Stratified 10-Fold Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Calculate the number of parameter combinations
# This is the sum of the product of parameters in each dict in param_grid
n_param_combinations = 0
for grid in param_grid:
    n_combinations = 1
    for param in grid:
        n_combinations *= len(grid[param])
    n_param_combinations += n_combinations

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit='roc_auc',  # Use 'roc_auc' to select the best model
    cv=cv,
    n_jobs=-1,  # Use all available cores
    verbose=0,  # Disable sklearn's verbose
    return_train_score=False
)

# Calculate the total number of parameter combinations for the progress bar
total_fits = int(
    sum(
        np.prod([len(values) for values in grid.values()])
        for grid in param_grid
    )
)

@contextmanager
def alive_joblib_bar(total):
    """
    Context manager to integrate alive-progress with joblib's Parallel processing.

    Parameters:
    - total: int, the total number of tasks to be processed.
    """
    with alive_bar(total, title='Grid Search Progress', bar='blocks', force_tty=True) as bar:
        # Store the original BatchCompletionCallBack.__call__ method
        original_callback = BatchCompletionCallBack.__call__
        lock = threading.Lock()

        def on_complete(self, *args, **kwargs):
            with lock:
                bar()
            return original_callback(self, *args, **kwargs)

        # Patch the BatchCompletionCallBack.__call__ method
        BatchCompletionCallBack.__call__ = on_complete
        try:
            yield
        finally:
            # Restore the original method to avoid side effects
            BatchCompletionCallBack.__call__ = original_callback

# Start the Grid Search with alive-progress
print("Starting Grid Search...")

with alive_joblib_bar(total_fits):
    grid_search.fit(X_selected, Y)

print("Grid Search Completed.")

# Retrieve the best average AUC and its standard deviation
best_auc = grid_search.best_score_
# Retrieve the standard deviation from cv_results_
# Identify the index of the best parameter set
best_index = grid_search.best_index_
best_auc_std = grid_search.cv_results_['std_test_roc_auc'][best_index]

# Retrieve the best hyperparameters
best_params = grid_search.best_params_

# Retrieve the other metrics for the best parameter set
best_accuracy = grid_search.cv_results_['mean_test_accuracy'][best_index]
best_accuracy_std = grid_search.cv_results_['std_test_accuracy'][best_index]

best_precision = grid_search.cv_results_['mean_test_precision'][best_index]
best_precision_std = grid_search.cv_results_['std_test_precision'][best_index]

best_f1 = grid_search.cv_results_['mean_test_f1'][best_index]
best_f1_std = grid_search.cv_results_['std_test_f1'][best_index]

# Output the results
print(f"\nBest Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
print(f"Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
print(f"Average Precision: {best_precision:.4f} ± {best_precision_std:.4f}")
print(f"Average F1 Score: {best_f1:.4f} ± {best_f1_std:.4f}")
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

Starting Grid Search...
Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▅▃▁ 369587/39936rid Search Progress |                                        | ▄▆█ 0/39936 [0%]Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▊                          | ▆▄▂ 13857/39936 

/home/azureuser/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▂▄▆ 369606/39936

/home/azureuser/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▇▅▃ 369623/39936

...

Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▄▆█ 389744/39936

/home/azureuser/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ █▆▄ 389761/39936

/home/azureuser/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ (!) 399360/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▂▂▄ 399360/39936
Grid Search Completed.

Best Average AUC: 0.7805 ± 0.0219
Average Accuracy: 0.8757 ± 0.0099
Average Precision: 0.3286 ± 0.0456
Average F1 Score: 0.3335 ± 0.0402

Best Hyperparameters:
  classifier__bootstrap: False
  classifier__class_weight: None
  classifier__criterion: entropy
  classifier__max_depth: None
  classifier__max_features: sqrt
  classifier__min_samples_leaf: 2
  classifier__min_samples_split: 2
  classifier__n_estimators: 250
  sampler: ADASYN()
  sampler__sampling_strategy: 0.5


In [2]:
from sklearn.model_selection import cross_val_predict

# Retrieve the best estimator from Grid Search
best_estimator = grid_search.best_estimator_

# Use cross_val_predict to get cross-validated predicted probabilities
# Setting method='predict_proba' and using cv to ensure consistency
print("\nGenerating cross-validated predicted probabilities...")
y_pred_proba = cross_val_predict(best_estimator, X_selected, Y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

# Define a range of threshold values to evaluate
thresholds = np.linspace(0.0, 1.0, 101)

# Initialize variables to store the best metrics and threshold
best_threshold = 0.5
best_f1_score = 0.0
best_accuracy = 0.0
best_precision = 0.0

print("Optimizing threshold to maximize F1 score...")

for threshold in thresholds:
    # Convert predicted probabilities to binary predictions based on the threshold
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate F1 score
    current_f1 = f1_score(Y, y_pred)
    
    # Update the best metrics and threshold if current F1 is better
    if current_f1 > best_f1_score:
        best_f1_score = current_f1
        best_threshold = threshold
        best_accuracy = accuracy_score(Y, y_pred)
        best_precision = precision_score(Y, y_pred, zero_division=0)

# Calculate standard deviations using cross-validation
# To compute standard deviations, we'll perform cross-validation predictions and calculate metrics at the best threshold

# Initialize lists to store per-fold metrics
f1_scores = []
accuracies = []
precisions = []

print("\nCalculating metrics at the optimal threshold across folds...")

for fold, (train_idx, test_idx) in enumerate(cv.split(X_selected, Y), 1):
    # Split data
    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    
    # Fit the model on the training data
    best_estimator.fit(X_train, y_train)
    
    # Predict probabilities on the test data
    y_proba_fold = best_estimator.predict_proba(X_test)[:, 1]
    
    # Apply the optimal threshold
    y_pred_fold = (y_proba_fold >= best_threshold).astype(int)
    
    # Calculate metrics
    fold_f1 = f1_score(y_test, y_pred_fold)
    fold_accuracy = accuracy_score(y_test, y_pred_fold)
    fold_precision = precision_score(y_test, y_pred_fold, zero_division=0)
    
    # Append to lists
    f1_scores.append(fold_f1)
    accuracies.append(fold_accuracy)
    precisions.append(fold_precision)
    
    print(f"  Fold {fold}: F1={fold_f1:.4f}, Accuracy={fold_accuracy:.4f}, Precision={fold_precision:.4f}")

# Calculate mean and standard deviation for the metrics
mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

# Output the optimized threshold and corresponding metrics
print(f"\n=== Optimized Threshold ===")
print(f"Threshold for Maximum F1 Score: {best_threshold:.2f}")

print(f"\n=== Metrics at Optimal Threshold ===")
print(f"F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"Precision: {mean_precision:.4f} ± {std_precision:.4f}")


Generating cross-validated predicted probabilities...
Optimizing threshold to maximize F1 score...

Calculating metrics at the optimal threshold across folds...
  Fold 1: F1=0.3550, Accuracy=0.8239, Precision=0.2679
  Fold 2: F1=0.3875, Accuracy=0.8414, Precision=0.2981
  Fold 3: F1=0.2976, Accuracy=0.8091, Precision=0.2232
  Fold 4: F1=0.3353, Accuracy=0.8139, Precision=0.2479
  Fold 5: F1=0.3182, Accuracy=0.8058, Precision=0.2333
  Fold 6: F1=0.2604, Accuracy=0.7977, Precision=0.1947
  Fold 7: F1=0.3333, Accuracy=0.8188, Precision=0.2500
  Fold 8: F1=0.3614, Accuracy=0.8285, Precision=0.2752
  Fold 9: F1=0.3095, Accuracy=0.8123, Precision=0.2342
  Fold 10: F1=0.3571, Accuracy=0.8252, Precision=0.2703

=== Optimized Threshold ===
Threshold for Maximum F1 Score: 0.26

=== Metrics at Optimal Threshold ===
F1 Score: 0.3315 ± 0.0348
Accuracy: 0.8177 ± 0.0120
Precision: 0.2495 ± 0.0283
